# Topic agenda calculations

This notebook creates the reusable calculation artifacts for the topic-agenda analysis. It computes article and discussion distributions, ranking-policy visible distributions, draw-level and story-level metrics, attention-decay sensitivity, concentration and exposure summaries, and both article-aware oracle benchmarks.

The expensive work is controlled by the two loops below. Each attention model has its own visible-distribution, metric, concentration, reference-target, and oracle caches. Each distance measure has its own oracle cache. By default, caches are reused only when their input and calculation-code signatures match. Set `ALLOW_OLDER_RUN_RESULTS=True` to explicitly reuse older calculations in the selected run's `analysis/` directory; their stored provenance is retained. `FORCE_RECOMPUTE=True` takes precedence and rebuilds calculations. Notebook 16 has the same older-results option for viewing historical outputs.

Run this notebook before [16_topic_agenda_analysis.ipynb](16_topic_agenda_analysis.ipynb).


Select the topic run in notebook 14, or pin `TOPIC_RUN_ID` below. Calculations and figures stay under the selected run's `analysis/` directory. Set `LEGACY_TOPIC_ROOT` explicitly to use an older unregistered fit.


In [ ]:
from pathlib import Path
import platform
import json

import numpy as np
import pandas as pd
import hashlib
from IPython.display import display

from commentgap_analysis.topic_modeling import (
    aggregate_topic_distributions,
)
from commentgap_analysis.topic_metrics import (
    build_topic_agenda_baseline_analysis,
    build_topic_agenda_rarefaction_analysis,
    compute_topic_metrics,
)
from commentgap_analysis.topic_policy import (
    aggregate_topic_policy_draw_distributions,
    aggregate_topic_policy_draw_metrics,
    build_topic_policy_concentration_analysis,
    build_topic_policy_exposure_coverage_analysis,
    build_policy_reference_target_metrics,
    read_existing_oracle,
    run_oracle_benchmark,
    weighted_policy_topic_distributions,
)
from commentgap_analysis.forum_scores import policy_specs

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'commentgap_analysis').exists() and (REPO_ROOT.parent / 'commentgap_analysis').exists():
    REPO_ROOT = REPO_ROOT.parent
from commentgap_analysis.topic_runs import resolve_run
RUNS_ROOT = REPO_ROOT / 'model_output/selection_2025/paper2_topic_runs'
TOPIC_RUN_ID = 'mcs15_nn10_ms5_seed2026_1e1b8fce5deb3e58'  # Final no-outlier-reduction model.
LEGACY_TOPIC_ROOT = None  # Set to the old artifact directory to explicitly use a legacy fit.
TOPIC_ROOT = (Path(LEGACY_TOPIC_ROOT) if LEGACY_TOPIC_ROOT is not None
              else resolve_run(RUNS_ROOT, TOPIC_RUN_ID))
print('Using topic model:', TOPIC_ROOT)
MEMBERSHIP_PATH = TOPIC_ROOT / 'document_topic_memberships.parquet'
COMMENTS_PATH = REPO_ROOT / 'model_output/selection_2025/paper2/analysis_comments.parquet'
OUTPUT_ROOT = TOPIC_ROOT / 'analysis'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TOPIC_POLICY_SEED = 2025
FORCE_RECOMPUTE = False
# Default: reuse only calculations with matching input/code signatures.
# Opt in to inspect older calculations without rebuilding the selected run.
ALLOW_OLDER_RUN_RESULTS = False
REUSE_EXISTING_RUN_RESULTS = ALLOW_OLDER_RUN_RESULTS and not FORCE_RECOMPUTE
if REUSE_EXISTING_RUN_RESULTS:
    print("Using older run calculations where available; stored provenance is retained.")
TOPIC_POLICY_N_JOBS = 1
TOPIC_POLICY_PROGRESS_EVERY = 25
TOPIC_POLICY_TIE_DRAWS = 10
TOPIC_POLICY_RANDOM_DRAWS = 25



from commentgap_analysis.topic_artifacts import (
    code_revision,
    compute_draw_metrics_in_batches,
    file_sha256,
    input_signature,
    package_versions,
    read_cached_parquet,
    reusable_artifacts,
    read_existing_run_parquet,
    signature_equal,
    begin_topic_run,
    publish_topic_run,
    topic_calculation_artifacts,
    topic_calculation_source_manifests,
    topic_calculation_config,
    write_signature,
)

# Invalidate any prior completion before fitting or writing calculation outputs.
RUN_TOKEN = begin_topic_run(OUTPUT_ROOT, {'run_id': TOPIC_RUN_ID})


## Development-set vote attention: primary spline and fitted sensitivities

The **smoothed empirical spline is the primary attention model**. The fitted power-law curve is a sensitivity model; the fitted exponential is retained only for the development fit comparison; the original fixed-parameter models are excluded.

Reconstruct **Rev. Chronological, Trees, Pinned** on the frozen development stories, using the eligible all-comment universe with at least 11 input comments. Pinning hides all descendants, including nested replies and replies linked through a missing parent. Hidden comments contribute neither ranks nor votes. Comments without valid topics still occupy visible ranks.

Each story contributes its visible upvotes + downvotes divided by its visible vote total. Average these fractions equally across stories, with absent ranks contributing zero. Zero-vote stories are excluded and counted. All fits model $p(r|L)=w_r/\sum_{j=1}^{L}w_j$, so the fit distinguishes rank weighting from how many stories reach each rank. The plotted fit lines average these length-conditioned predictions, not the unnormalised weights.

The spline models log weights with a cubic B-spline on log rank, denser fixed knots near the top, and a second-difference coefficient penalty. Smoothing is selected by inner development-fold validation; outer frozen development folds compare all three models using equal-story cross entropy (lower is better). Final models are refitted on all development stories. The spline is primary by design, regardless of this comparison. The final smoothing CV score is not substituted for the nested out-of-fold comparison.

For application, freeze the fitted weights and renormalise over each policy's visible length. Beyond the largest development rank, the spline holds its last weight constant; the parametric families continue their fitted curves. The sparse tail and extrapolation are assumptions, not measurements. Votes remain a voting-activity proxy, confounded by age, content, pinning and historical positions; a flexible fit does not identify causal attention.


In [ ]:
from commentgap_analysis.vote_attention import fit_vote_attention_models
from commentgap_analysis.vote_attention_plotting import plot_vote_attention_curve

# Fitted attention is itself a reusable artifact of the selected run.
fit_paths = {
    'curve': OUTPUT_ROOT / 'topic_policy_vote_attention_curve.csv',
    'fit': OUTPUT_ROOT / 'topic_policy_vote_attention_fit.json',
    'comparison': OUTPUT_ROOT / 'topic_policy_vote_attention_comparison.csv',
    'cv': OUTPUT_ROOT / 'topic_policy_vote_attention_cv.csv',
    'tuning': OUTPUT_ROOT / 'topic_policy_vote_attention_tuning.csv',
}
from commentgap_analysis.topic_runs import file_inventory
SPLIT_PATH = REPO_ROOT / 'model_output/selection_2025/model_data/master_article_split.parquet'
CHOICE_PATH = REPO_ROOT / 'model_output/selection_2025/model_data/choice_set_all.parquet'
RAW_COMMENTS_ROOT = REPO_ROOT / 'data/scrape_2025/comments/year=2025'
fit_signature = {
    'split_sha256': file_sha256(SPLIT_PATH),
    'choice_sha256': file_sha256(CHOICE_PATH),
    'raw_comments_inventory': file_inventory(RAW_COMMENTS_ROOT),
    'fitting_code_sha256': file_sha256(REPO_ROOT / 'commentgap_analysis/vote_attention.py'),
    'policy_code_sha256': file_sha256(REPO_ROOT / 'commentgap_analysis/forum_scores.py'),
    'packages': package_versions(['numpy', 'pandas', 'scipy']),
}
fit_manifest_path = OUTPUT_ROOT / 'topic_policy_vote_attention_cache_manifest.json'
can_reuse_fit = reusable_artifacts(
    fit_paths.values(), manifest_path=fit_manifest_path, signature=fit_signature,
    force_recompute=FORCE_RECOMPUTE, allow_older_results=ALLOW_OLDER_RUN_RESULTS,
)
if can_reuse_fit:
    vote_attention_curve = pd.read_csv(fit_paths['curve'])
    vote_attention_fit = json.loads(fit_paths['fit'].read_text())
    fit_comparison = pd.read_csv(fit_paths['comparison'])
    fit_cv = pd.read_csv(fit_paths['cv'])
    fit_tuning = pd.read_csv(fit_paths['tuning'])
    if not {'spline', 'power_law'}.issubset(vote_attention_fit.get('models', {})):
        can_reuse_fit = False
        print('Existing attention fit is incomplete; refitting')
    else:
        print('Reusing fitted vote attention from selected run')
if not can_reuse_fit:
    import pyarrow.dataset as ds
    from commentgap_analysis.forum_scores import STRUCTURAL_COLUMNS, PRIMARY_MIN_COMMENTS
    article_split = pd.read_parquet(SPLIT_PATH)
    article_split['story_id'] = article_split.story_id.astype(str)
    development_ids = set(article_split.loc[article_split.split_role.eq('development'), 'story_id'])
    test_ids = set(article_split.loc[article_split.split_role.eq('paper2_test'), 'story_id'])
    assert development_ids and not development_ids.intersection(test_ids)
    choice = pd.read_parquet(CHOICE_PATH, columns=['story_id', 'comment_id', 'n_candidates'])
    for key in ('story_id', 'comment_id'):
        choice[key] = choice[key].astype(str)
    development_keys = choice.loc[
        choice.story_id.isin(development_ids) & choice.n_candidates.ge(PRIMARY_MIN_COMMENTS),
        ['story_id', 'comment_id'],
    ]
    raw_dataset = ds.dataset(RAW_COMMENTS_ROOT, format='parquet', partitioning=None)
    development_comments = raw_dataset.to_table(
        columns=list(STRUCTURAL_COLUMNS),
        filter=ds.field('story_id').isin(sorted(set(development_keys.story_id))),
    ).to_pandas()
    for key in ('story_id', 'comment_id'):
        development_comments[key] = development_comments[key].astype(str)
    development_comments = development_keys.merge(
        development_comments, on=['story_id', 'comment_id'], how='left', validate='one_to_one', indicator=True,
    )
    assert development_comments['_merge'].eq('both').all(), 'Missing development comment structure'
    development_comments = development_comments.drop(columns='_merge')
    assert not set(development_comments.story_id).intersection(test_ids)
    (vote_attention_curve, vote_attention_fit, fit_comparison, fit_cv, fit_tuning) = fit_vote_attention_models(
        development_comments, article_split, min_comments=PRIMARY_MIN_COMMENTS,
    )
    vote_attention_fit['provenance'] = {
        'split_sha256': file_sha256(SPLIT_PATH), 'choice_sha256': file_sha256(CHOICE_PATH),
        'development_comments_sha256': hashlib.sha256(
            pd.util.hash_pandas_object(development_comments.sort_values(['story_id', 'comment_id']), index=False).values.tobytes()
        ).hexdigest(),
        'fitting_code_sha256': file_sha256(REPO_ROOT / 'commentgap_analysis/vote_attention.py'),
        'policy_code_sha256': file_sha256(REPO_ROOT / 'commentgap_analysis/forum_scores.py'),
    }
    vote_attention_curve.to_csv(fit_paths['curve'], index=False)
    write_signature(fit_paths['fit'], vote_attention_fit)
    fit_comparison.to_csv(fit_paths['comparison'], index=False)
    fit_cv.to_csv(fit_paths['cv'], index=False)
    fit_tuning.to_csv(fit_paths['tuning'], index=False)
    write_signature(fit_manifest_path, fit_signature)
display(fit_comparison.round(6))
display(pd.DataFrame([
    {'model': model, **{key: config.get(key) for key in ('parameter', 'smoothing', 'fit_at_boundary')}}
    for model, config in vote_attention_fit['models'].items()
]))
fit_labels = {
    'spline': 'Smoothed empirical spline (primary)',
    'power_law': f"Fitted power law (alpha={vote_attention_fit['models']['power_law']['parameter']:.3f})",
    'exponential': f"Fitted exponential (lambda={vote_attention_fit['models']['exponential']['parameter']:.5f})",
}
plot_vote_attention_curve(
    vote_attention_curve, vote_attention_fit,
    OUTPUT_ROOT / 'topic_policy_vote_attention_curve.png',
)
if not can_reuse_fit:
    del development_comments, choice


## Topic memberships and baseline calculations


In [ ]:
import pyarrow.parquet as pq
if not MEMBERSHIP_PATH.exists():
    raise FileNotFoundError(f'Run notebook 14 first: {MEMBERSHIP_PATH}')
INPUT_SIGNATURE = input_signature(
    repo_root=REPO_ROOT, membership_path=MEMBERSHIP_PATH,
    comments_path=COMMENTS_PATH, topic_root=TOPIC_ROOT,
)

topic_columns = [
    field.name for field in pq.read_schema(MEMBERSHIP_PATH)
    if field.name.startswith('topic_') and field.name[6:].isdigit()
]
membership_columns = ['story_id', 'comment_id', 'doc_type', 'valid_topic', *topic_columns]
memberships = pd.read_parquet(MEMBERSHIP_PATH, columns=membership_columns)
for column in topic_columns:
    memberships[column] = memberships[column].astype('float32')

comment_columns = [
    'story_id', 'comment_id', 'is_root', 'root_comment_id', 'parent_comment_id',
    'created_at', 'preorder_position', 'display_order', 'root_order', 'is_sticky',
    'votes_positive', 'votes_negative', 'relative_votes',
    'regression_audience_score', 'regression_editor_score',
    'xgb_metadata_audience_score', 'xgb_metadata_editor_score',
    'xgb_metadata_text_audience_score', 'xgb_metadata_text_editor_score',
    'neural_metadata_audience_score', 'neural_metadata_editor_score',
    'neural_metadata_text_audience_score', 'neural_metadata_text_editor_score',
]
comments = pd.read_parquet(COMMENTS_PATH, columns=comment_columns)
for frame in (memberships, comments):
    frame['story_id'] = frame['story_id'].astype(str)
    frame['comment_id'] = frame['comment_id'].astype(str)

comment_memberships = memberships.loc[memberships.doc_type.eq('comment'), ['story_id', 'comment_id']]
assert not comment_memberships.duplicated().any()
assert not comments.duplicated(['story_id', 'comment_id']).any()
comment_keys = pd.MultiIndex.from_frame(comments[['story_id', 'comment_id']])
assert pd.MultiIndex.from_frame(comment_memberships).isin(comment_keys).all(), 'Ranking comments missing topic memberships'
assert memberships.doc_type.eq('comment').any(), 'No comment topic memberships found'

article_distributions = aggregate_topic_distributions(
    memberships[memberships.doc_type.eq('article')], group_columns=('story_id', 'doc_type')
).drop(columns='doc_type')
discussion_distributions = aggregate_topic_distributions(
    memberships[memberships.doc_type.eq('comment')], group_columns=('story_id', 'doc_type')
).drop(columns='doc_type')

baseline_paths = {
    'story': OUTPUT_ROOT / 'topic_agenda_baseline_story_metrics.parquet',
    'summary': OUTPUT_ROOT / 'topic_agenda_baseline_summary.csv',
}
baseline_manifest_path = OUTPUT_ROOT / 'topic_agenda_baseline_cache_manifest.json'
baseline_signature = {'inputs': INPUT_SIGNATURE, 'topic_columns': list(topic_columns)}
if reusable_artifacts(
    baseline_paths.values(), manifest_path=baseline_manifest_path, signature=baseline_signature,
    force_recompute=FORCE_RECOMPUTE, allow_older_results=ALLOW_OLDER_RUN_RESULTS,
):
    baseline_analysis = {
        'story': pd.read_parquet(baseline_paths['story']),
        'summary': pd.read_csv(baseline_paths['summary']),
    }
    print('Reusing baseline agenda calculations from selected run')
else:
    baseline_analysis = build_topic_agenda_baseline_analysis(
        article_distributions, discussion_distributions, topic_columns=topic_columns, output_root=OUTPUT_ROOT,
    )
    write_signature(baseline_manifest_path, baseline_signature)
rarefaction_paths = {
    'story': OUTPUT_ROOT / 'topic_agenda_rarefaction_story_metrics.parquet',
    'summary': OUTPUT_ROOT / 'topic_agenda_rarefaction_summary.csv',
}
rarefaction_manifest_path = OUTPUT_ROOT / 'topic_agenda_rarefaction_cache_manifest.json'
rarefaction_signature = {'inputs': INPUT_SIGNATURE, 'n_draws': 100, 'seed': TOPIC_POLICY_SEED}
if reusable_artifacts(
    rarefaction_paths.values(), manifest_path=rarefaction_manifest_path, signature=rarefaction_signature,
    force_recompute=FORCE_RECOMPUTE, allow_older_results=ALLOW_OLDER_RUN_RESULTS,
):
    rarefaction_analysis = {
        'story': pd.read_parquet(rarefaction_paths['story']),
        'summary': pd.read_csv(rarefaction_paths['summary']),
    }
    print('Reusing rarefaction calculations from selected run')
else:
    rarefaction_analysis = build_topic_agenda_rarefaction_analysis(
        memberships, n_draws=100, seed=TOPIC_POLICY_SEED, output_root=OUTPUT_ROOT,
    )
    write_signature(rarefaction_manifest_path, rarefaction_signature)
display(baseline_analysis['summary'].round(4))
display(rarefaction_analysis['summary'].round(4))

## Policy configuration and restartable metric batching

The policy factorial is held fixed across attention models. The draw-level visible distributions are retained so nonlinear metrics can be computed per presentation draw and then averaged within story. The batch writer uses atomic part files and reports progress and ETA; an interrupted run resumes from completed parts.

In [ ]:
SCORE_POLICIES = {
    'chronological': None, 'reverse_chronological': None, 'relative_votes': None, 'upvotes': None,
    'random': None,
    'regression_audience': 'regression_audience_score', 'regression_editor': 'regression_editor_score',
    'xgb_metadata_audience': 'xgb_metadata_audience_score', 'xgb_metadata_editor': 'xgb_metadata_editor_score',
    'xgb_metadata_text_audience': 'xgb_metadata_text_audience_score', 'xgb_metadata_text_editor': 'xgb_metadata_text_editor_score',
    'neural_metadata_audience': 'neural_metadata_audience_score', 'neural_metadata_editor': 'neural_metadata_editor_score',
    'neural_metadata_text_audience': 'neural_metadata_text_audience_score', 'neural_metadata_text_editor': 'neural_metadata_text_editor_score',
}
POLICY_SPECS = policy_specs()
assert len(POLICY_SPECS) == 90

PRIMARY_ATTENTION_MODEL = 'spline'
ANALYSIS_MODELS = ('spline', 'power_law')
ATTENTION_MODELS = {
    model: {
        **config, 'prefix': f'vote_{model}_', 'label': fit_labels[model],
        'role': 'primary' if model == PRIMARY_ATTENTION_MODEL else 'sensitivity',
        'fit_provenance': vote_attention_fit['provenance'],
    }
    for model, config in vote_attention_fit['models'].items()
    if model in ANALYSIS_MODELS
}
DISTANCE_MEASURES = {
    'cosine': {'objective': 'cosine_similarity', 'label': 'raw-proportion cosine'},
    'jensen_shannon': {'objective': 'jensen_shannon_distance', 'label': 'Jensen-Shannon distance'},
}
ORACLE_STRUCTURED_START_POLICIES = tuple(name for name in SCORE_POLICIES if name != 'random')
ORACLE_KWARGS = {
    'max_iterations': 5, 'n_starts': 8, 'random_state': TOPIC_POLICY_SEED,
    'n_perturbations': 4, 'perturbation_fraction': 0.10, 'n_local_moves': 2,
    'allow_invalid_placement': True, 'exact_max_valid_comments': 8,
    'progress_every': TOPIC_POLICY_PROGRESS_EVERY,
}


print('Stories with article and discussion distributions:', len(article_distributions.merge(discussion_distributions, on='story_id')))

## Attention-model loop

This is the only policy-distribution and metric calculation loop. It first reuses a validated visible-distribution cache when available, then reuses or builds draw-level metrics, aggregates them within story, and writes the compact story-level metrics consumed by notebook 16. The concentration and exposure summaries are also written here because they are derived calculation artifacts rather than presentation-only tables.

In [ ]:
comment_memberships_frame = memberships.loc[memberships.doc_type.eq('comment')].copy()

for attention_model, attention in ATTENTION_MODELS.items():
    prefix = attention['prefix']
    mode = attention['rank_weight_mode']
    visible_path = OUTPUT_ROOT / f'topic_policy_{prefix}visible_distributions.parquet'
    metrics_draws_path = OUTPUT_ROOT / f'topic_policy_{prefix}metrics_draws.parquet'
    metrics_path = OUTPUT_ROOT / f'topic_policy_{prefix}metrics.parquet'
    manifest_path = OUTPUT_ROOT / f'topic_policy_{prefix}cache_manifest.json'
    metrics_manifest_path = OUTPUT_ROOT / f'topic_policy_{prefix}metrics_cache_manifest.json'
    signature = {
        'cache_version': 3, 'attention_model': attention_model, 'attention': attention,
        'inputs': INPUT_SIGNATURE, 'topic_columns': list(topic_columns),
        'policy_specs': [repr(spec) for spec in POLICY_SPECS], 'score_policies': SCORE_POLICIES,
        'tie_draws': TOPIC_POLICY_TIE_DRAWS, 'random_draws': TOPIC_POLICY_RANDOM_DRAWS,
        'seed': TOPIC_POLICY_SEED, 'code_revision': code_revision(),
        'code_hash': file_sha256(REPO_ROOT / 'commentgap_analysis/topic_modeling.py'),
        'policy_code_hash': file_sha256(REPO_ROOT / 'commentgap_analysis/forum_scores.py'),
    }
    visible_draws = read_existing_run_parquet(
        visible_path, required_columns=('story_id', 'policy_id', 'draw', *topic_columns),
        reuse_existing=REUSE_EXISTING_RUN_RESULTS,
    )
    if visible_draws is None:
        visible_draws = read_cached_parquet(
            visible_path, manifest_path=manifest_path, signature=signature,
            required_columns=('story_id', 'policy_id', 'draw', *topic_columns),
        ) if not FORCE_RECOMPUTE else None
    if visible_draws is None:
        print(f'Building {attention["label"]} visible distributions')
        visible_draws = weighted_policy_topic_distributions(
            comment_memberships_frame, comments, tie_draws=TOPIC_POLICY_TIE_DRAWS,
            random_draws=TOPIC_POLICY_RANDOM_DRAWS, seed=TOPIC_POLICY_SEED,
            rank_weight_power=attention['rank_weight_power'], rank_weight_mode=mode,
            rank_decay=attention['rank_decay'], rank_weight_values=attention['rank_weight_values'], progress=True,
            progress_every_stories=TOPIC_POLICY_PROGRESS_EVERY, n_jobs=TOPIC_POLICY_N_JOBS,
            retain_draws=True,
        )
        visible_draws.to_parquet(visible_path, index=False)
        write_signature(manifest_path, signature)
    else:
        print(f'Reusing {attention["label"]} visible distributions: {len(visible_draws):,} rows')

    metrics_draws = read_existing_run_parquet(
        metrics_draws_path,
        required_columns=('story_id', 'policy_id', 'draw', 'article_visible_js_distance', 'cosine_progress'),
        reuse_existing=REUSE_EXISTING_RUN_RESULTS,
    )
    if metrics_draws is None:
        metrics_draws = None if FORCE_RECOMPUTE else read_cached_parquet(
            metrics_draws_path, manifest_path=metrics_manifest_path, signature=signature,
            required_columns=('story_id', 'policy_id', 'draw', 'article_visible_js_distance', 'cosine_progress'),
        )
    if metrics_draws is None:
        # Resume parts only for exactly these inputs and fitted parameters.
        parts_path = metrics_draws_path.with_suffix('.parts')
        parts_manifest_path = parts_path / 'manifest.json'
        if FORCE_RECOMPUTE or not signature_equal(parts_manifest_path, signature):
            if parts_path.exists():
                for stale_part in parts_path.glob('part_*.parquet'):
                    stale_part.unlink()
            parts_path.mkdir(parents=True, exist_ok=True)
            write_signature(parts_manifest_path, signature)
        compute_draw_metrics_in_batches(
            visible_path, metrics_draws_path,
            compute_metrics=lambda batch: compute_topic_metrics(
                article_distributions, discussion_distributions, batch
            ),
        )
        metrics_draws = pd.read_parquet(metrics_draws_path)
        write_signature(metrics_manifest_path, signature)
    else:
        print(f'Reusing {attention["label"]} draw metrics: {len(metrics_draws):,} rows')
    metrics = aggregate_topic_policy_draw_metrics(metrics_draws)
    metrics.to_parquet(metrics_path, index=False)
    visible_distributions = aggregate_topic_policy_draw_distributions(visible_draws)

    coverage = build_topic_policy_exposure_coverage_analysis(metrics, output_root=OUTPUT_ROOT, output_prefix=prefix)
    concentration = build_topic_policy_concentration_analysis(
        visible_distributions, article_distributions, discussion_distributions, metrics,
        topic_columns=topic_columns, output_root=OUTPUT_ROOT, output_prefix=prefix,
    )
    reference_targets = build_policy_reference_target_metrics(
        visible_draws, article_distributions, metrics, topic_columns=topic_columns,
    )
    reference_targets.to_parquet(OUTPUT_ROOT / f'{prefix}topic_policy_reference_target_metrics.parquet', index=False)

    oracle_results = {}
    for distance_measure, distance in DISTANCE_MEASURES.items():
        oracle_prefix = f'topic_policy_{prefix}cosine_' if distance_measure == 'cosine' else f'topic_policy_{prefix}js_'
        stem = oracle_prefix + 'oracle'
        policy_id = f'oracle_{attention_model}_{distance_measure}'
        paths = {
            'visible_path': OUTPUT_ROOT / f'{stem}_visible_distributions.parquet',
            'metrics_path': OUTPUT_ROOT / f'{stem}_metrics.parquet',
            'summary_path': OUTPUT_ROOT / f'{stem}_summary.csv',
            'skipped_path': OUTPUT_ROOT / f'{stem}_skipped.csv',
            'exact_validation_path': OUTPUT_ROOT / f'{stem}_exact_validation.csv',
        }
        existing_oracle = read_existing_oracle(
            paths, topic_columns=topic_columns, reuse_existing=REUSE_EXISTING_RUN_RESULTS,
        )
        if existing_oracle is not None:
            oracle_results[distance_measure] = existing_oracle
            print(f'Reusing {attention["label"]} {distance["label"]} oracle from selected run')
            continue
        oracle_results[distance_measure] = run_oracle_benchmark(
            memberships, comments, article_distributions, discussion_distributions, topic_columns,
            policy_id=policy_id, ordering=f'{attention_model}_{distance_measure}_oracle',
            objective=distance['objective'], structured_start_policies=ORACLE_STRUCTURED_START_POLICIES,
            force_recompute=FORCE_RECOMPUTE, rank_weight_power=attention['rank_weight_power'],
            rank_weight_mode=mode, rank_decay=attention['rank_decay'],
            rank_weight_values=attention['rank_weight_values'], label=f'{attention["label"]} {distance["label"]} oracle',
            **paths, **ORACLE_KWARGS,
        )

    run_metadata = {
        'attention_model': attention_model, 'attention': attention,
        'n_visible_draw_rows': int(len(visible_draws)), 'n_metric_draw_rows': int(len(metrics_draws)),
        'n_metric_rows': int(len(metrics)), 'n_story_rows': int(metrics.story_id.nunique()),
        'distance_measures': list(DISTANCE_MEASURES), 'requested_input_signature': INPUT_SIGNATURE,
        'allow_older_run_results': REUSE_EXISTING_RUN_RESULTS,
        'source_cache_manifests': {
            path.name: json.loads(path.read_text()) if path.exists() else None
            for path in (manifest_path, metrics_manifest_path, fit_manifest_path)
        },
        'code_hash': signature['code_hash'], 'code_revision': signature['code_revision'],
        'platform': platform.platform(), 'python': platform.python_version(),
        'packages': package_versions(['bertopic', 'umap-learn', 'hdbscan', 'scikit-learn', 'pandas', 'numpy']),
    }
    (OUTPUT_ROOT / f'topic_policy_{prefix}run_metadata.json').write_text(json.dumps(run_metadata, indent=2, sort_keys=True, default=str) + '\n')
    print(f'Completed {attention_model}: {len(metrics):,} story-policy rows; {len(metrics_draws):,} draw-policy rows')

In [ ]:
# Compare fitted sensitivities against the primary spline, within policy cells.
attention_summaries = []
for attention_model, attention in ATTENTION_MODELS.items():
    frame = pd.read_parquet(OUTPUT_ROOT / f"topic_policy_{attention['prefix']}metrics.parquet")
    summary = frame.groupby(['ordering', 'reply_mode', 'pinned'], as_index=False).agg(
        js_distance=('article_visible_js_distance', 'mean'),
        alignment_gain=('alignment_gain', 'mean'),
    )
    summary['attention_model'] = attention_model
    attention_summaries.append(summary)
attention_sensitivity = pd.concat(attention_summaries, ignore_index=True)
primary = attention_sensitivity.loc[attention_sensitivity.attention_model.eq(PRIMARY_ATTENTION_MODEL)].drop(columns='attention_model')
attention_sensitivity = attention_sensitivity.merge(
    primary, on=['ordering', 'reply_mode', 'pinned'], suffixes=('', '_primary'), validate='many_to_one',
)
attention_sensitivity['change_in_js_distance'] = attention_sensitivity.js_distance - attention_sensitivity.js_distance_primary
attention_sensitivity['change_in_alignment_gain'] = attention_sensitivity.alignment_gain - attention_sensitivity.alignment_gain_primary
attention_sensitivity.to_csv(OUTPUT_ROOT / 'topic_policy_fitted_attention_sensitivity.csv', index=False)
display(attention_sensitivity.round(4))


## Integrity checks

The calculation run must retain one row per story, policy cell, and draw in the draw-level artifacts, and all 90 factorial policy cells must be represented in the story-level metrics. Notebook 16 reads only the saved artifacts below.

In [ ]:
for attention_model, attention in ATTENTION_MODELS.items():
    prefix = attention['prefix']
    metrics = pd.read_parquet(OUTPUT_ROOT / f'topic_policy_{prefix}metrics.parquet')
    draws = pd.read_parquet(OUTPUT_ROOT / f'topic_policy_{prefix}metrics_draws.parquet')
    assert not metrics.duplicated(['story_id', 'policy_id']).any()
    assert not draws.duplicated(['story_id', 'policy_id', 'draw']).any()
    expected = {spec.policy_id for spec in POLICY_SPECS}
    assert expected.issubset(set(metrics.policy_id.astype(str)))
    assert (OUTPUT_ROOT / f'{prefix}topic_policy_reference_target_metrics.parquet').exists()
    for distance_measure in DISTANCE_MEASURES:
        stem = f'topic_policy_{prefix}cosine_' if distance_measure == 'cosine' else f'topic_policy_{prefix}js_'
        assert (OUTPUT_ROOT / f'{stem}oracle_metrics.parquet').exists()
    print(attention_model, 'checks passed:', len(metrics), 'story-policy rows;', len(draws), 'draw rows')

In [ ]:
CALCULATION_CONFIG = topic_calculation_config(
    score_policies=SCORE_POLICIES, policy_specs=POLICY_SPECS,
    analysis_models=tuple(ATTENTION_MODELS), distance_measures=tuple(DISTANCE_MEASURES),
    seed=TOPIC_POLICY_SEED, tie_draws=TOPIC_POLICY_TIE_DRAWS, random_draws=TOPIC_POLICY_RANDOM_DRAWS,
    oracle_kwargs=ORACLE_KWARGS,
)
CALCULATION_CODE = {
    'topic_artifacts_code_hash': file_sha256(REPO_ROOT / 'commentgap_analysis/topic_artifacts.py'),
    'topic_modeling_code_hash': file_sha256(REPO_ROOT / 'commentgap_analysis/topic_modeling.py'),
    'topic_metrics_code_hash': file_sha256(REPO_ROOT / 'commentgap_analysis/topic_metrics.py'),
    'topic_policy_code_hash': file_sha256(REPO_ROOT / 'commentgap_analysis/topic_policy.py'),
    'forum_scores_code_hash': file_sha256(REPO_ROOT / 'commentgap_analysis/forum_scores.py'),
    'vote_attention_code_hash': file_sha256(REPO_ROOT / 'commentgap_analysis/vote_attention.py'),
    'code_revision': code_revision(),
}
CALCULATION_PROVENANCE = {
    'run_id': TOPIC_RUN_ID, 'inputs': INPUT_SIGNATURE, 'config': CALCULATION_CONFIG,
    'code': CALCULATION_CODE,
    'reuse': {
        'allow_older_run_results': REUSE_EXISTING_RUN_RESULTS,
        'source_cache_manifests_preserved': True,
    },
}
publish_topic_run(
    OUTPUT_ROOT, run_token=RUN_TOKEN, provenance=CALCULATION_PROVENANCE,
    required_artifacts=topic_calculation_artifacts(
        OUTPUT_ROOT, analysis_models=tuple(ATTENTION_MODELS),
        distance_measures=tuple(DISTANCE_MEASURES),
    ),
    optional_artifacts=topic_calculation_source_manifests(
        OUTPUT_ROOT, analysis_models=tuple(ATTENTION_MODELS),
        distance_measures=tuple(DISTANCE_MEASURES),
    ),
)
print('Published complete topic calculation manifest')
